# German-French Electricity Data Preprocessing

**Research question:** How are raw SMARD price records and Open-Meteo ERA5
weather records cleaned and combined into a single hourly modelling dataframe?

This notebook is the preprocessing walkthrough. Every stage calls the
repository preprocessing API under `energy_trading_pipeline.preprocessing`
and prints the dataframe it produced, so the effect of each transformation is
visible row by row: raw records, normalized timestamps, cleaned records, the
combined energy and weather table, and the final spread dataset.


## Reproducibility boundary

The default `real` mode reads the cached 2024 SMARD and Open-Meteo ERA5
Parquet artifacts. Set `ETP_NOTEBOOK_MODE=fixture` for the small offline
repository fixtures used by automated verification. Neither mode downloads
data, uses credentials, or reaches the network.

Every stage prints `rows x columns` followed by the leading rows of the
dataframe. Set `ETP_NOTEBOOK_PREVIEW_ROWS` to change how many rows appear.
Notebook 01 explores the data; this notebook performs and documents the
transformations.


In [1]:
import json
import tempfile
from pathlib import Path
import os

import pandas as pd

from energy_trading_pipeline.config.loader import load_yaml_file
from energy_trading_pipeline.data_ingestion.local_loader import load_local_data
from energy_trading_pipeline.preprocessing.alignment import align_hourly_data
from energy_trading_pipeline.preprocessing.cleaning import clean_records
from energy_trading_pipeline.preprocessing.spread import (
    calculate_spread,
    save_processed_data,
)
from energy_trading_pipeline.preprocessing.timestamps import normalize_timestamps
from energy_trading_pipeline.preprocessing.validation import validate_hourly_index

working_directory = Path.cwd()
if (working_directory / "pyproject.toml").is_file():
    REPO_ROOT = working_directory
elif (working_directory.parent / "pyproject.toml").is_file():
    REPO_ROOT = working_directory.parent
else:
    raise FileNotFoundError(
        "Run this notebook from the repository root or notebooks directory"
    )
MODE = os.environ.get("ETP_NOTEBOOK_MODE", "real").strip().lower()
if MODE not in {"real", "fixture"}:
    raise ValueError("ETP_NOTEBOOK_MODE must be 'real' or 'fixture'")
PREVIEW_ROWS = int(os.environ.get("ETP_NOTEBOOK_PREVIEW_ROWS", "5"))

REAL_PATHS = {
    "prices": REPO_ROOT / "data/processed/run_20260911_053845/prices_de_fr.parquet",
    "weather_locations": REPO_ROOT / "data/processed/run_20260913_132539/weather_2024_locations.parquet",
    "weather_aggregates": REPO_ROOT / "data/processed/run_20260913_132539/weather_2024_country_aggregates.parquet",
}
FIXTURE_PATHS = {
    "price_de": REPO_ROOT / "tests/fixtures/sample_prices_de.csv",
    "price_fr": REPO_ROOT / "tests/fixtures/sample_prices_fr.csv",
    "weather": REPO_ROOT / "tests/fixtures/sample_weather.csv",
}

def show(value):
    # Keep notebook display optional for the pure-Python fixture smoke test.
    try:
        get_ipython
    except NameError:
        if isinstance(value, pd.DataFrame):
            print(value.to_string())
        else:
            print(value)
    else:
        from IPython.display import display
        display(value)

def preview(label, frame, rows=None):
    # Print the shape and leading rows of a preprocessing stage dataframe.
    row_count = PREVIEW_ROWS if rows is None else rows
    print(f"\n{label}: {len(frame):,} rows x {len(frame.columns)} columns")
    show(frame.head(row_count))

config = load_yaml_file(REPO_ROOT / "configs/experiment.yaml")
CONFIG_CLEANING = config["data"]["cleaning"]
# The notebook demonstrates duplicate removal instead of aborting on it, so the
# declared duplicate policy is relaxed here and the difference is printed.
APPLIED_CLEANING = {
    "duplicate_policy": "keep_first",
    "required_missing_policy": CONFIG_CLEANING["required_missing_policy"],
}
print(f"Execution mode: {MODE}")
show(
    pd.DataFrame(
        [
            {
                "policy": name,
                "configs/experiment.yaml": CONFIG_CLEANING[name],
                "applied_in_notebook": APPLIED_CLEANING[name],
            }
            for name in APPLIED_CLEANING
        ]
    )
)


Execution mode: real


,policy,configs/experiment.yaml,applied_in_notebook
0,duplicate_policy,raise,keep_first
1,required_missing_policy,raise,raise


## Stage 1 - Raw source records

Prices are hourly day-ahead values from Bundesnetzagentur SMARD (upstream
source: ENTSO-E) in EUR/MWh, with `price_de` the Germany/Luxembourg bidding
zone and `price_fr` France. Weather is Open-Meteo ERA5 reanalysis for four
representative German and four French locations, published both per location
and as documented country aggregates. Nothing has been touched yet.


In [2]:
selected_paths = REAL_PATHS if MODE == "real" else FIXTURE_PATHS
missing_paths = [str(path) for path in selected_paths.values() if not path.is_file()]
if missing_paths:
    raise FileNotFoundError(
        "Notebook inputs are missing for " + MODE + " mode:\n- " + "\n- ".join(missing_paths)
    )

source_frames = {
    dataset: load_local_data(path) for dataset, path in selected_paths.items()
}
provenance = pd.DataFrame(
    [
        {
            "dataset": dataset,
            "path": str(selected_paths[dataset].relative_to(REPO_ROOT)),
            "rows": len(frame),
            "columns": len(frame.columns),
        }
        for dataset, frame in source_frames.items()
    ]
)
show(provenance)

for dataset, frame in source_frames.items():
    preview(f"stage 1 - raw {dataset}", frame)


,dataset,path,rows,columns
0,prices,data/processed/run_20260911_053845/prices_de_f...,8784,3
1,weather_locations,data/processed/run_20260913_132539/weather_202...,8784,33
2,weather_aggregates,data/processed/run_20260913_132539/weather_202...,8784,9



stage 1 - raw prices: 8,784 rows x 3 columns


,timestamp,price_de,price_fr
0,2023-12-31 23:00:00+00:00,0.10,0.10
1,2024-01-01 00:00:00+00:00,0.01,0.01
2,2024-01-01 01:00:00+00:00,0.00,0.00
3,2024-01-01 02:00:00+00:00,-0.01,-0.01
4,2024-01-01 03:00:00+00:00,-0.03,-0.03



stage 1 - raw weather_locations: 8,784 rows x 33 columns


,timestamp,temperature_2m_c_de_oldenburg,wind_speed_10m_m_s_de_oldenburg,wind_speed_100m_m_s_de_oldenburg,shortwave_radiation_w_m2_de_oldenburg,temperature_2m_c_de_husum,wind_speed_10m_m_s_de_husum,wind_speed_100m_m_s_de_husum,shortwave_radiation_w_m2_de_husum,temperature_2m_c_de_potsdam,...,wind_speed_100m_m_s_fr_reims,shortwave_radiation_w_m2_fr_reims,temperature_2m_c_fr_bordeaux,wind_speed_10m_m_s_fr_bordeaux,wind_speed_100m_m_s_fr_bordeaux,shortwave_radiation_w_m2_fr_bordeaux,temperature_2m_c_fr_toulouse,wind_speed_10m_m_s_fr_toulouse,wind_speed_100m_m_s_fr_toulouse,shortwave_radiation_w_m2_fr_toulouse
0,2023-12-31 23:00:00+00:00,7.4,5.85,10.32,0.0,7.0,5.80,10.35,0.0,5.9,...,12.57,0.0,7.9,3.14,6.56,0.0,6.8,3.68,7.67,0.0
1,2024-01-01 00:00:00+00:00,7.5,5.82,10.18,0.0,6.8,5.42,9.80,0.0,5.7,...,12.04,0.0,7.7,2.98,6.31,0.0,6.5,3.58,7.62,0.0
2,2024-01-01 01:00:00+00:00,7.4,5.80,10.06,0.0,6.7,5.08,9.31,0.0,5.5,...,11.71,0.0,7.8,2.98,6.27,0.0,6.5,3.58,7.68,0.0
3,2024-01-01 02:00:00+00:00,7.8,6.09,10.40,0.0,6.6,5.14,9.25,0.0,5.2,...,11.50,0.0,7.8,3.28,6.62,0.0,6.5,3.45,7.34,0.0
4,2024-01-01 03:00:00+00:00,7.7,6.01,10.32,0.0,6.8,5.39,9.53,0.0,5.0,...,11.72,0.0,8.0,3.32,6.72,0.0,6.5,3.02,6.78,0.0



stage 1 - raw weather_aggregates: 8,784 rows x 9 columns


,timestamp,temperature_2m_c_de_mean,wind_speed_10m_m_s_de_weighted,wind_speed_100m_m_s_de_weighted,shortwave_radiation_w_m2_de_weighted,temperature_2m_c_fr_mean,wind_speed_10m_m_s_fr_weighted,wind_speed_100m_m_s_fr_weighted,shortwave_radiation_w_m2_fr_weighted
0,2023-12-31 23:00:00+00:00,6.350,5.110378,9.392372,0.0,7.550,7.098519,11.792888,0.0
1,2024-01-01 00:00:00+00:00,6.200,5.112666,9.345341,0.0,7.325,6.903333,11.505465,0.0
2,2024-01-01 01:00:00+00:00,6.075,5.019476,9.142426,0.0,7.250,6.795334,11.294331,0.0
3,2024-01-01 02:00:00+00:00,6.125,5.112360,9.213914,0.0,7.175,6.681287,11.132536,0.0
4,2024-01-01 03:00:00+00:00,6.125,5.218010,9.349438,0.0,7.150,6.944552,11.399044,0.0


## Stage 2 - What actually needs cleaning

Quality is measured before any change so the cleaning decisions that follow
are justified by evidence rather than assumed. Duplicate timestamps, internal
hourly gaps, off-hour values, ordering, and missing cells are counted per
dataset. The fixture price file carries a deliberate duplicated hour.


In [3]:
quality_rows = []
for dataset, frame in source_frames.items():
    parsed = pd.to_datetime(frame["timestamp"], utc=True)
    hourly = validate_hourly_index(parsed)
    missing_by_column = frame.isna().sum()
    quality_rows.append(
        {
            "dataset": dataset,
            "rows": len(frame),
            "chronological": hourly["is_sorted"],
            "duplicate_timestamp_rows": hourly["duplicate_count"],
            "missing_internal_hours": hourly["missing_hour_count"],
            "off_hour_timestamps": hourly["non_hourly_count"],
            "missing_cells": int(missing_by_column.sum()),
            "columns_with_missing": ", ".join(
                sorted(column for column, count in missing_by_column.items() if count)
            ),
        }
    )
pre_clean_quality = pd.DataFrame(quality_rows)
show(pre_clean_quality)


,dataset,rows,chronological,duplicate_timestamp_rows,missing_internal_hours,off_hour_timestamps,missing_cells,columns_with_missing
0,prices,8784,True,0,0,0,0,
1,weather_locations,8784,True,0,0,0,0,
2,weather_aggregates,8784,True,0,0,0,0,


## Stage 3 - Timestamp normalization

`normalize_timestamps` parses every timestamp individually, converts it to UTC,
and sorts chronologically. Mixed naive and aware values never silently assume
UTC, ambiguous or nonexistent local clock times raise, and numeric epochs are
rejected. No row is added, removed, or filled here - only parsed and ordered.


In [4]:
normalized_frames = {}
timestamp_reports = {}
for dataset, frame in source_frames.items():
    normalized, timestamp_report = normalize_timestamps(frame, source_timezone="UTC")
    normalized_frames[dataset] = normalized
    timestamp_reports[dataset] = timestamp_report

timestamp_audit = pd.DataFrame(
    [
        {
            "dataset": dataset,
            "source_timezone": report["source_timezone"],
            "normalized_timezone": report["normalized_timezone"],
            "naive_timestamps": report["naive_timestamp_count"],
            "sorted": report["hourly_validation"]["is_sorted"],
            "duplicate_rows": report["hourly_validation"]["duplicate_count"],
            "missing_hours": report["hourly_validation"]["missing_hour_count"],
            "dtype": str(normalized_frames[dataset]["timestamp"].dtype),
        }
        for dataset, report in timestamp_reports.items()
    ]
)
show(timestamp_audit)

for dataset, frame in normalized_frames.items():
    preview(f"stage 3 - normalized {dataset}", frame)


,dataset,source_timezone,normalized_timezone,naive_timestamps,sorted,duplicate_rows,missing_hours,dtype
0,prices,UTC,UTC,0,True,0,0,"datetime64[ms, UTC]"
1,weather_locations,UTC,UTC,0,True,0,0,"datetime64[us, UTC]"
2,weather_aggregates,UTC,UTC,0,True,0,0,"datetime64[us, UTC]"



stage 3 - normalized prices: 8,784 rows x 3 columns


,timestamp,price_de,price_fr
0,2023-12-31 23:00:00+00:00,0.10,0.10
1,2024-01-01 00:00:00+00:00,0.01,0.01
2,2024-01-01 01:00:00+00:00,0.00,0.00
3,2024-01-01 02:00:00+00:00,-0.01,-0.01
4,2024-01-01 03:00:00+00:00,-0.03,-0.03



stage 3 - normalized weather_locations: 8,784 rows x 33 columns


,timestamp,temperature_2m_c_de_oldenburg,wind_speed_10m_m_s_de_oldenburg,wind_speed_100m_m_s_de_oldenburg,shortwave_radiation_w_m2_de_oldenburg,temperature_2m_c_de_husum,wind_speed_10m_m_s_de_husum,wind_speed_100m_m_s_de_husum,shortwave_radiation_w_m2_de_husum,temperature_2m_c_de_potsdam,...,wind_speed_100m_m_s_fr_reims,shortwave_radiation_w_m2_fr_reims,temperature_2m_c_fr_bordeaux,wind_speed_10m_m_s_fr_bordeaux,wind_speed_100m_m_s_fr_bordeaux,shortwave_radiation_w_m2_fr_bordeaux,temperature_2m_c_fr_toulouse,wind_speed_10m_m_s_fr_toulouse,wind_speed_100m_m_s_fr_toulouse,shortwave_radiation_w_m2_fr_toulouse
0,2023-12-31 23:00:00+00:00,7.4,5.85,10.32,0.0,7.0,5.80,10.35,0.0,5.9,...,12.57,0.0,7.9,3.14,6.56,0.0,6.8,3.68,7.67,0.0
1,2024-01-01 00:00:00+00:00,7.5,5.82,10.18,0.0,6.8,5.42,9.80,0.0,5.7,...,12.04,0.0,7.7,2.98,6.31,0.0,6.5,3.58,7.62,0.0
2,2024-01-01 01:00:00+00:00,7.4,5.80,10.06,0.0,6.7,5.08,9.31,0.0,5.5,...,11.71,0.0,7.8,2.98,6.27,0.0,6.5,3.58,7.68,0.0
3,2024-01-01 02:00:00+00:00,7.8,6.09,10.40,0.0,6.6,5.14,9.25,0.0,5.2,...,11.50,0.0,7.8,3.28,6.62,0.0,6.5,3.45,7.34,0.0
4,2024-01-01 03:00:00+00:00,7.7,6.01,10.32,0.0,6.8,5.39,9.53,0.0,5.0,...,11.72,0.0,8.0,3.32,6.72,0.0,6.5,3.02,6.78,0.0



stage 3 - normalized weather_aggregates: 8,784 rows x 9 columns


,timestamp,temperature_2m_c_de_mean,wind_speed_10m_m_s_de_weighted,wind_speed_100m_m_s_de_weighted,shortwave_radiation_w_m2_de_weighted,temperature_2m_c_fr_mean,wind_speed_10m_m_s_fr_weighted,wind_speed_100m_m_s_fr_weighted,shortwave_radiation_w_m2_fr_weighted
0,2023-12-31 23:00:00+00:00,6.350,5.110378,9.392372,0.0,7.550,7.098519,11.792888,0.0
1,2024-01-01 00:00:00+00:00,6.200,5.112666,9.345341,0.0,7.325,6.903333,11.505465,0.0
2,2024-01-01 01:00:00+00:00,6.075,5.019476,9.142426,0.0,7.250,6.795334,11.294331,0.0
3,2024-01-01 02:00:00+00:00,6.125,5.112360,9.213914,0.0,7.175,6.681287,11.132536,0.0
4,2024-01-01 03:00:00+00:00,6.125,5.218010,9.349438,0.0,7.150,6.944552,11.399044,0.0


## Stage 4 - Cleaning rules

`clean_records` applies explicit policies and appends one audit record per
dataset to `preprocessing_log.jsonl`. Duplicate instants are resolved first by
stable `keep_first`, then required columns are checked: a missing required
price fails the run. Optional weather columns containing any missing value are
excluded entirely with a warning. Nothing is imputed and no hour is invented,
so a cleaned dataframe only ever loses rows or columns, never gains them.


In [5]:
audit_workspace = tempfile.TemporaryDirectory(prefix="etp_preprocessing_")
audit_dir = Path(audit_workspace.name) / "run_notebook"

if MODE == "real":
    required_map = {
        "prices": ["price_de", "price_fr"],
        "weather_locations": [],
        "weather_aggregates": [],
    }
else:
    required_map = {
        "price_de": ["price_de"],
        "price_fr": ["price_fr"],
        "weather": [],
    }

cleaned_frames = {}
cleanup_reports = {}
for dataset, frame in normalized_frames.items():
    required_columns = required_map[dataset]
    optional_columns = [
        column
        for column in frame.columns
        if column != "timestamp" and column not in required_columns
    ]
    cleaned, cleanup_report = clean_records(
        frame,
        required_columns=required_columns,
        optional_columns=optional_columns,
        duplicate_policy=APPLIED_CLEANING["duplicate_policy"],
        required_missing_policy=APPLIED_CLEANING["required_missing_policy"],
        run_dir=audit_dir,
        dataset=dataset,
    )
    cleaned_frames[dataset] = cleaned
    cleanup_reports[dataset] = cleanup_report

cleanup_audit = pd.DataFrame(
    [
        {
            "dataset": dataset,
            "input_rows": report["input_rows"],
            "output_rows": report["output_rows"],
            "duplicates_removed": report["duplicate_rows_removed"],
            "required_rows_removed": report["required_rows_removed"],
            "columns_kept": len(report["retained_columns"]),
            "excluded_optional_columns": ", ".join(report["excluded_optional_columns"]),
        }
        for dataset, report in cleanup_reports.items()
    ]
)
show(cleanup_audit)
print(f"\nAudit record written to: {audit_dir / 'preprocessing_log.jsonl'}")

for dataset, frame in cleaned_frames.items():
    preview(f"stage 4 - cleaned {dataset}", frame)


,dataset,input_rows,output_rows,duplicates_removed,required_rows_removed,columns_kept,excluded_optional_columns
0,prices,8784,8784,0,0,3,
1,weather_locations,8784,8784,0,0,33,
2,weather_aggregates,8784,8784,0,0,9,



Audit record written to: /var/folders/lw/2wsyf7pd3p3f3h68czszyft40000gn/T/etp_preprocessing_8crf1q2_/run_notebook/preprocessing_log.jsonl

stage 4 - cleaned prices: 8,784 rows x 3 columns


,timestamp,price_de,price_fr
0,2023-12-31 23:00:00+00:00,0.10,0.10
1,2024-01-01 00:00:00+00:00,0.01,0.01
2,2024-01-01 01:00:00+00:00,0.00,0.00
3,2024-01-01 02:00:00+00:00,-0.01,-0.01
4,2024-01-01 03:00:00+00:00,-0.03,-0.03



stage 4 - cleaned weather_locations: 8,784 rows x 33 columns


,timestamp,temperature_2m_c_de_oldenburg,wind_speed_10m_m_s_de_oldenburg,wind_speed_100m_m_s_de_oldenburg,shortwave_radiation_w_m2_de_oldenburg,temperature_2m_c_de_husum,wind_speed_10m_m_s_de_husum,wind_speed_100m_m_s_de_husum,shortwave_radiation_w_m2_de_husum,temperature_2m_c_de_potsdam,...,wind_speed_100m_m_s_fr_reims,shortwave_radiation_w_m2_fr_reims,temperature_2m_c_fr_bordeaux,wind_speed_10m_m_s_fr_bordeaux,wind_speed_100m_m_s_fr_bordeaux,shortwave_radiation_w_m2_fr_bordeaux,temperature_2m_c_fr_toulouse,wind_speed_10m_m_s_fr_toulouse,wind_speed_100m_m_s_fr_toulouse,shortwave_radiation_w_m2_fr_toulouse
0,2023-12-31 23:00:00+00:00,7.4,5.85,10.32,0.0,7.0,5.80,10.35,0.0,5.9,...,12.57,0.0,7.9,3.14,6.56,0.0,6.8,3.68,7.67,0.0
1,2024-01-01 00:00:00+00:00,7.5,5.82,10.18,0.0,6.8,5.42,9.80,0.0,5.7,...,12.04,0.0,7.7,2.98,6.31,0.0,6.5,3.58,7.62,0.0
2,2024-01-01 01:00:00+00:00,7.4,5.80,10.06,0.0,6.7,5.08,9.31,0.0,5.5,...,11.71,0.0,7.8,2.98,6.27,0.0,6.5,3.58,7.68,0.0
3,2024-01-01 02:00:00+00:00,7.8,6.09,10.40,0.0,6.6,5.14,9.25,0.0,5.2,...,11.50,0.0,7.8,3.28,6.62,0.0,6.5,3.45,7.34,0.0
4,2024-01-01 03:00:00+00:00,7.7,6.01,10.32,0.0,6.8,5.39,9.53,0.0,5.0,...,11.72,0.0,8.0,3.32,6.72,0.0,6.5,3.02,6.78,0.0



stage 4 - cleaned weather_aggregates: 8,784 rows x 9 columns


,timestamp,temperature_2m_c_de_mean,wind_speed_10m_m_s_de_weighted,wind_speed_100m_m_s_de_weighted,shortwave_radiation_w_m2_de_weighted,temperature_2m_c_fr_mean,wind_speed_10m_m_s_fr_weighted,wind_speed_100m_m_s_fr_weighted,shortwave_radiation_w_m2_fr_weighted
0,2023-12-31 23:00:00+00:00,6.350,5.110378,9.392372,0.0,7.550,7.098519,11.792888,0.0
1,2024-01-01 00:00:00+00:00,6.200,5.112666,9.345341,0.0,7.325,6.903333,11.505465,0.0
2,2024-01-01 01:00:00+00:00,6.075,5.019476,9.142426,0.0,7.250,6.795334,11.294331,0.0
3,2024-01-01 02:00:00+00:00,6.125,5.112360,9.213914,0.0,7.175,6.681287,11.132536,0.0
4,2024-01-01 03:00:00+00:00,6.125,5.218010,9.349438,0.0,7.150,6.944552,11.399044,0.0


### Duplicate removal, row by row

Where a duplicated hour was removed, the retained and discarded records are
printed side by side. This is the only place rows disappear in `real` mode
without a required-value failure, so it is shown explicitly rather than
summarized.


In [6]:
duplicate_examples = []
for dataset, report in cleanup_reports.items():
    if not report["duplicate_rows_removed"]:
        continue
    normalized = normalized_frames[dataset]
    duplicated_hours = normalized.loc[
        normalized["timestamp"].duplicated(keep=False), "timestamp"
    ].unique()
    for hour in duplicated_hours:
        candidates = normalized[normalized["timestamp"] == hour].copy()
        candidates.insert(0, "dataset", dataset)
        candidates.insert(
            1,
            "decision",
            ["kept (first)"] + ["dropped"] * (len(candidates) - 1),
        )
        duplicate_examples.append(candidates)

if duplicate_examples:
    show(pd.concat(duplicate_examples, ignore_index=True))
else:
    print("No duplicate timestamps were present in any dataset.")


No duplicate timestamps were present in any dataset.


## Stage 5 - Combining energy and weather into one dataframe

This is the join that produces a single hourly table. Price records supply the
required columns; weather records are exogenous and joined on the UTC hour.

In `fixture` mode the repository API `align_hourly_data` builds the hourly
timeline from the configured inclusive calendar dates and reindexes each source
onto it. In `real` mode the cached artifacts follow a local-market calendar
expressed in UTC rather than a UTC calendar year, so exact timestamp equality
is asserted first and the join is a validated one-to-one merge - no boundary
hour is added or dropped. Location-level weather stays out of the joined table;
only the documented country aggregates are carried forward.


In [7]:
if MODE == "real":
    price_timestamps = pd.DatetimeIndex(cleaned_frames["prices"]["timestamp"])
    weather_timestamps = pd.DatetimeIndex(
        cleaned_frames["weather_aggregates"]["timestamp"]
    )
    location_timestamps = pd.DatetimeIndex(
        cleaned_frames["weather_locations"]["timestamp"]
    )
    timestamps_match_exactly = price_timestamps.equals(
        weather_timestamps
    ) and price_timestamps.equals(location_timestamps)
    if not timestamps_match_exactly:
        raise ValueError(
            "Price, location-weather, and aggregate-weather timestamps do not match exactly"
        )
    combined_data = cleaned_frames["prices"].merge(
        cleaned_frames["weather_aggregates"],
        on="timestamp",
        how="inner",
        validate="one_to_one",
    )
    join_report = {
        "operation": "exact_timestamp_equality_and_one_to_one_merge",
        "energy_rows": len(cleaned_frames["prices"]),
        "weather_rows": len(cleaned_frames["weather_aggregates"]),
        "timestamps_match_exactly": timestamps_match_exactly,
        "weather_columns_joined": len(cleaned_frames["weather_aggregates"].columns) - 1,
        "weather_columns_held_back": len(cleaned_frames["weather_locations"].columns) - 1,
        "output_rows": len(combined_data),
    }
else:
    combined_data, join_report = align_hourly_data(
        cleaned_frames["price_de"],
        cleaned_frames["price_fr"],
        weather=cleaned_frames["weather"],
        start_date=config["dates"]["start_date"],
        end_date=config["dates"]["end_date"],
    )
    join_report = {
        "operation": "align_hourly_data",
        "date_range": f"{join_report['date_range']['start']} to {join_report['date_range']['end']}",
        "energy_rows": len(cleaned_frames["price_de"]),
        "weather_rows": len(cleaned_frames["weather"]),
        "excluded_optional_columns": ", ".join(join_report["excluded_optional_columns"]),
        "output_rows": join_report["output_rows"],
    }

show(pd.DataFrame([join_report]).T.rename(columns={0: "value"}))
preview("stage 5 - combined energy + weather", combined_data)


,value
operation,exact_timestamp_equality_and_one_to_one_merge
energy_rows,8784
weather_rows,8784
timestamps_match_exactly,True
weather_columns_joined,8
weather_columns_held_back,32
output_rows,8784



stage 5 - combined energy + weather: 8,784 rows x 11 columns


,timestamp,price_de,price_fr,temperature_2m_c_de_mean,wind_speed_10m_m_s_de_weighted,wind_speed_100m_m_s_de_weighted,shortwave_radiation_w_m2_de_weighted,temperature_2m_c_fr_mean,wind_speed_10m_m_s_fr_weighted,wind_speed_100m_m_s_fr_weighted,shortwave_radiation_w_m2_fr_weighted
0,2023-12-31 23:00:00+00:00,0.10,0.10,6.350,5.110378,9.392372,0.0,7.550,7.098519,11.792888,0.0
1,2024-01-01 00:00:00+00:00,0.01,0.01,6.200,5.112666,9.345341,0.0,7.325,6.903333,11.505465,0.0
2,2024-01-01 01:00:00+00:00,0.00,0.00,6.075,5.019476,9.142426,0.0,7.250,6.795334,11.294331,0.0
3,2024-01-01 02:00:00+00:00,-0.01,-0.01,6.125,5.112360,9.213914,0.0,7.175,6.681287,11.132536,0.0
4,2024-01-01 03:00:00+00:00,-0.03,-0.03,6.125,5.218010,9.349438,0.0,7.150,6.944552,11.399044,0.0


### Where every combined column came from

Each column of the joined table is traced back to the cleaned dataset that
contributed it, together with its dtype and remaining missing count. Every
missing count must be zero: the cleaning and join policies exclude incomplete
columns rather than carrying gaps forward.


In [8]:
column_provenance = pd.DataFrame(
    [
        {
            "column": column,
            "source_dataset": ", ".join(
                dataset
                for dataset, frame in cleaned_frames.items()
                if column in frame.columns
            )
            or "derived",
            "role": (
                "join key"
                if column == "timestamp"
                else "energy" if column.startswith("price_") else "weather"
            ),
            "dtype": str(combined_data[column].dtype),
            "missing_values": int(combined_data[column].isna().sum()),
        }
        for column in combined_data.columns
    ]
)
show(column_provenance)


,column,source_dataset,role,dtype,missing_values
0,timestamp,"prices, weather_locations, weather_aggregates",join key,"datetime64[ms, UTC]",0
1,price_de,prices,energy,float64,0
2,price_fr,prices,energy,float64,0
3,temperature_2m_c_de_mean,weather_aggregates,weather,float64,0
4,wind_speed_10m_m_s_de_weighted,weather_aggregates,weather,float64,0
5,wind_speed_100m_m_s_de_weighted,weather_aggregates,weather,float64,0
6,shortwave_radiation_w_m2_de_weighted,weather_aggregates,weather,float64,0
7,temperature_2m_c_fr_mean,weather_aggregates,weather,float64,0
8,wind_speed_10m_m_s_fr_weighted,weather_aggregates,weather,float64,0
9,wind_speed_100m_m_s_fr_weighted,weather_aggregates,weather,float64,0


## Stage 6 - Spread target

`calculate_spread` re-sorts on UTC, revalidates that the hourly index is
unique and contiguous, and computes `spread = price_de - price_fr` in float64.
Negative prices and negative spreads are valid market outcomes and are kept.


In [9]:
prepared_data = calculate_spread(combined_data)

preview("stage 6 - prepared_data (head)", prepared_data)
print("\nstage 6 - prepared_data (tail)")
show(prepared_data.tail(PREVIEW_ROWS))

print("\nSpread arithmetic on the first rows")
show(
    prepared_data[["timestamp", "price_de", "price_fr", "spread"]]
    .head(PREVIEW_ROWS)
    .assign(recomputed=lambda frame: frame["price_de"] - frame["price_fr"])
)



stage 6 - prepared_data (head): 8,784 rows x 12 columns


,timestamp,price_de,price_fr,temperature_2m_c_de_mean,wind_speed_10m_m_s_de_weighted,wind_speed_100m_m_s_de_weighted,shortwave_radiation_w_m2_de_weighted,temperature_2m_c_fr_mean,wind_speed_10m_m_s_fr_weighted,wind_speed_100m_m_s_fr_weighted,shortwave_radiation_w_m2_fr_weighted,spread
0,2023-12-31 23:00:00+00:00,0.10,0.10,6.350,5.110378,9.392372,0.0,7.550,7.098519,11.792888,0.0,0.0
1,2024-01-01 00:00:00+00:00,0.01,0.01,6.200,5.112666,9.345341,0.0,7.325,6.903333,11.505465,0.0,0.0
2,2024-01-01 01:00:00+00:00,0.00,0.00,6.075,5.019476,9.142426,0.0,7.250,6.795334,11.294331,0.0,0.0
3,2024-01-01 02:00:00+00:00,-0.01,-0.01,6.125,5.112360,9.213914,0.0,7.175,6.681287,11.132536,0.0,0.0
4,2024-01-01 03:00:00+00:00,-0.03,-0.03,6.125,5.218010,9.349438,0.0,7.150,6.944552,11.399044,0.0,0.0



stage 6 - prepared_data (tail)


,timestamp,price_de,price_fr,temperature_2m_c_de_mean,wind_speed_10m_m_s_de_weighted,wind_speed_100m_m_s_de_weighted,shortwave_radiation_w_m2_de_weighted,temperature_2m_c_fr_mean,wind_speed_10m_m_s_fr_weighted,wind_speed_100m_m_s_fr_weighted,shortwave_radiation_w_m2_fr_weighted,spread
8779,2024-12-31 18:00:00+00:00,67.77,92.78,2.075,7.541705,12.246921,0.0,2.900,5.010857,8.393610,0.0,-25.01
8780,2024-12-31 19:00:00+00:00,35.56,79.04,2.125,7.775503,12.645486,0.0,2.800,5.328519,8.815511,0.0,-43.48
8781,2024-12-31 20:00:00+00:00,15.70,50.11,2.225,8.019402,12.992576,0.0,2.700,5.532646,9.163468,0.0,-34.41
8782,2024-12-31 21:00:00+00:00,9.06,63.36,2.375,8.151774,13.212602,0.0,2.950,5.621443,9.480841,0.0,-54.30
8783,2024-12-31 22:00:00+00:00,0.52,60.18,2.550,7.250493,12.569782,0.0,3.175,5.362211,9.344276,0.0,-59.66



Spread arithmetic on the first rows


,timestamp,price_de,price_fr,spread,recomputed
0,2023-12-31 23:00:00+00:00,0.10,0.10,0.0,0.0
1,2024-01-01 00:00:00+00:00,0.01,0.01,0.0,0.0
2,2024-01-01 01:00:00+00:00,0.00,0.00,0.0,0.0
3,2024-01-01 02:00:00+00:00,-0.01,-0.01,0.0,0.0
4,2024-01-01 03:00:00+00:00,-0.03,-0.03,0.0,0.0


## Stage 7 - Final validation

The handoff dataframe is checked directly rather than trusted: hourly index
integrity, UTC timezone, no missing values anywhere, and the spread identity.
These assertions fail the notebook if a preprocessing stage regresses.


In [10]:
hourly_validation = validate_hourly_index(prepared_data["timestamp"])
final_audit = {
    "mode": MODE,
    "rows": len(prepared_data),
    "columns": len(prepared_data.columns),
    "first_timestamp": prepared_data["timestamp"].iloc[0].isoformat(),
    "last_timestamp": prepared_data["timestamp"].iloc[-1].isoformat(),
    "timezone": str(prepared_data["timestamp"].dt.tz),
    "hourly_index_valid": hourly_validation["is_hourly"],
    "duplicate_timestamps": hourly_validation["duplicate_count"],
    "missing_hours": hourly_validation["missing_hour_count"],
    "missing_values": int(prepared_data.isna().sum().sum()),
    "spread_identity_holds": bool(
        (
            (prepared_data["spread"] - (prepared_data["price_de"] - prepared_data["price_fr"])).abs()
            < 1e-9
        ).all()
    ),
}
assert final_audit["hourly_index_valid"]
assert final_audit["timezone"] == "UTC"
assert final_audit["missing_values"] == 0
assert final_audit["spread_identity_holds"]
show(pd.DataFrame([final_audit]).T.rename(columns={0: "value"}))


,value
mode,real
rows,8784
columns,12
first_timestamp,2023-12-31T23:00:00+00:00
last_timestamp,2024-12-31T22:00:00+00:00
timezone,UTC
hourly_index_valid,True
duplicate_timestamps,0
missing_hours,0
missing_values,0


## Stage 8 - Persisting the processed artifact

`save_processed_data` writes the Parquet table plus a `.metadata.yaml` sidecar
recording source files, UTC date range, target column, and the join metadata.
This notebook writes to a temporary directory so running it never overwrites a
canonical artifact under `data/processed/`; the pipeline CLI writes the real
one to the configured `processed_data_parquet_path`. The saved file is read
back and printed to confirm the round trip.


In [ ]:
output_workspace = tempfile.TemporaryDirectory(prefix="etp_preprocessing_output_")
output_path = Path(output_workspace.name) / "prices.parquet"
if MODE == "real":
    source_files = {
        "price_de": REAL_PATHS["prices"],
        "price_fr": REAL_PATHS["prices"],
        "weather": REAL_PATHS["weather_aggregates"],
    }
else:
    source_files = {
        "price_de": FIXTURE_PATHS["price_de"],
        "price_fr": FIXTURE_PATHS["price_fr"],
        "weather": FIXTURE_PATHS["weather"],
    }

parquet_path, metadata_path = save_processed_data(
    prepared_data,
    output_path,
    source_files=source_files,
    alignment_metadata=join_report,
)
print(f"Parquet:  {parquet_path}")
print(f"Metadata: {metadata_path}")
print(f"\n{metadata_path.name}\n{metadata_path.read_text(encoding='utf-8')}")

reloaded_data = load_local_data(parquet_path)
preview("stage 8 - reloaded from Parquet", reloaded_data)
assert len(reloaded_data) == len(prepared_data)
assert reloaded_data.columns.tolist() == prepared_data.columns.tolist()


## Stage summary

One table recording what each stage did to the row and column counts, for the
whole pipeline at a glance.


In [ ]:
stage_summary = pd.DataFrame(
    [
        {
            "stage": "1 - raw",
            "dataframe": dataset,
            "rows": len(frame),
            "columns": len(frame.columns),
        }
        for dataset, frame in source_frames.items()
    ]
    + [
        {
            "stage": "3 - normalized",
            "dataframe": dataset,
            "rows": len(frame),
            "columns": len(frame.columns),
        }
        for dataset, frame in normalized_frames.items()
    ]
    + [
        {
            "stage": "4 - cleaned",
            "dataframe": dataset,
            "rows": len(frame),
            "columns": len(frame.columns),
        }
        for dataset, frame in cleaned_frames.items()
    ]
    + [
        {
            "stage": "5 - combined",
            "dataframe": "combined_data",
            "rows": len(combined_data),
            "columns": len(combined_data.columns),
        },
        {
            "stage": "6 - prepared",
            "dataframe": "prepared_data",
            "rows": len(prepared_data),
            "columns": len(prepared_data.columns),
        },
    ]
)
show(stage_summary)


## Limitations and handoff

- Cleaning never imputes. Missing required prices fail the run; optional
  weather columns with gaps are dropped whole, so a column can vanish between
  the raw and cleaned previews above.
- The `real` join asserts exact timestamp equality instead of reindexing onto a
  UTC calendar year, because the cached artifacts follow the local-market year.
- ERA5 values are same-hour reanalysis, not archived day-ahead forecasts. They
  must be lagged or replaced with properly vintaged forecasts before
  leakage-safe backtesting.
- Location-level weather is cleaned and audited but deliberately not joined;
  only documented country aggregates enter `prepared_data`.

`prepared_data` is the synchronized hourly dataframe handed to feature
engineering. No feature is engineered and no model is trained here.
